💡 **Environment:** `clamp-analyses` (R kernel)

# Is log2(x + 1) needed before CLAMP?

**What we did.** Before CLAMP, every expression matrix is filtered (mean ≥ 0.5, variance ≥ 0.1) and z-scored per gene. The bulk compendia are log2(x + 1)-transformed first (`CLAMP::cleanFBM`, called by `preprocessCLAMPFBM` for GTEx and recount2 and directly in `scripts/archs4/preprocess.R`); pseudobulk CPM is not (`CLAMP::preprocessCLAMP` in `scripts/pseudobulk/preprocess.R`). We compared the linear values with log2(x + 1) using three standard diagnostics, on the genes each pipeline keeps (2,000 random samples per compendium, all samples per pseudobulk dataset; `scripts/expression_scale/sample_linear.R`, rule `expression_scale_sample`):

- `vsn::meanSdPlot`: gene SD against the rank of gene mean; a flat trend means every gene varies about the same.
- `edgeR::plotMeanVar`: gene variance against mean (log-log axes, linear values). Crosses on the dotted line (variance = mean²) mean each gene varies by the same percentage of its expression, which only a log evens out; on the solid Poisson line (variance = mean) a milder correction would do.
- `e1071::skewness` per gene: above 2, a few samples dominate a gene once it is z-scored.

**Result.**

| Data | Input | log2(x + 1) | Needed? |
|---|---|---|---|
| GTEx | TPM | yes | yes: variance ~ mean² (slope 1.88); 64% of genes strongly skewed, 10% after log |
| recount2 | RPKM | yes | yes: slope 1.83; 99% → 10% |
| ARCHS4 | TPM | yes | yes: slope 1.87; 99% → 16% |
| Pseudobulk (6 datasets) | CPM | no | not needed: slope 1.4–1.7 (close to Poisson at low expression); 7–40% of genes skewed, median skewness 0.7–1.7 |

Summing many cells per donor makes pseudobulk CPM far better behaved than bulk TPM/RPKM, so it is used on the linear scale; the bulk compendia require log2(x + 1).

In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(edgeR); library(vsn); library(e1071); library(ggplot2); library(patchwork)
})
if (exists("snakemake")) {
  matrices <- snakemake@input[["matrices"]]
  out <- snakemake@output
} else {
  OUT <- here::here("output/01_model_building/04_expression_scale")
  matrices <- file.path(OUT, "linear_matrices.rds")
  out <- list(summary = file.path(OUT, "summary.csv"), mean_sd = file.path(OUT, "meanSdPlot.png"),
              mean_var = file.path(OUT, "plotMeanVar.png"), skewness = file.path(OUT, "skewness.png"))
}
lin <- readRDS(matrices)
log2_applied <- setNames(grepl("TPM|RPKM", names(lin)), names(lin))
data.table(dataset = names(lin), genes = sapply(lin, nrow), samples = sapply(lin, ncol), log2_applied = log2_applied)

## vsn::meanSdPlot

Top: linear values (log y axis); bottom: log2(x + 1). A rising red line means highly expressed genes vary much more than lowly expressed ones; after log2(x + 1) the line is roughly flat for every dataset.

In [ ]:
msd <- function(x, title, ylog) {
  p <- meanSdPlot(x, plot = FALSE)$gg + ggtitle(title) + theme_bw(base_size = 7) + theme(legend.position = "none")
  if (ylog) p + scale_y_log10() else p
}
top <- lapply(names(lin), function(n) msd(lin[[n]], paste0(n, "\nlinear"), TRUE))
bottom <- lapply(names(lin), function(n) msd(log2(lin[[n]] + 1), "log2(x + 1)", FALSE))
p_msd <- wrap_plots(c(top, bottom), nrow = 2)
ggsave(out[["mean_sd"]], p_msd, width = 22, height = 5.5, dpi = 150)
options(repr.plot.width = 22, repr.plot.height = 5.5)
p_msd

## edgeR::plotMeanVar (linear values)

Each red cross is a bin of genes with similar expression (average mean and variance on the linear scale; axes are log). Solid line: Poisson, variance = mean. Dotted line: variance = mean², each gene varying by the same percentage of its expression, which a log turns into equal variance. The slope of the crosses is reported per dataset. Libraries with no values (four of the sampled recount2 libraries are entirely missing in the input and set to 0, as in `cleanFBM`) are dropped here.

In [ ]:
png(out[["mean_var"]], width = 22, height = 2.8, units = "in", res = 150)
par(mfrow = c(1, length(lin)), mar = c(4, 4, 3, 1), cex = 0.6)
slopes <- sapply(names(lin), function(n) {
  x <- lin[[n]][rowSums(lin[[n]]) > 0, colSums(lin[[n]]) > 0]
  mv <- plotMeanVar(DGEList(x), main = n)
  abline(0, 2, lty = 3)
  ok <- mv$avemeans > 0 & mv$avevars > 0
  unname(coef(lm(log10(mv$avevars[ok]) ~ log10(mv$avemeans[ok])))[2])
})
invisible(dev.off())
IRdisplay::display_png(file = out[["mean_var"]])
round(slopes, 2)

## e1071::skewness per gene

Moment skewness (`type = 1`) of each gene across samples, linear vs log2(x + 1); dashed line at 2 (strongly right-skewed: a few samples dominate the gene after z-scoring).

In [ ]:
sk <- rbindlist(lapply(names(lin), function(n) rbind(
  data.table(dataset = n, scale = "linear", skew = apply(lin[[n]], 1, skewness, type = 1)),
  data.table(dataset = n, scale = "log2(x + 1)", skew = apply(log2(lin[[n]] + 1), 1, skewness, type = 1)))))
sk <- sk[is.finite(skew)]
sk[, dataset := factor(dataset, names(lin))]
p <- ggplot(sk, aes(skew, fill = scale)) +
  geom_histogram(bins = 60, position = "identity", alpha = 0.6) +
  geom_vline(xintercept = 2, linetype = 2) +
  facet_wrap(~dataset, nrow = 1, scales = "free_y") +
  coord_cartesian(xlim = c(-2, 15)) +
  scale_fill_manual(values = c(linear = "#eb6834", "log2(x + 1)" = "#2a78d6")) +
  labs(x = "per-gene skewness (e1071::skewness)", y = "genes", fill = NULL) +
  theme_bw(base_size = 7) + theme(legend.position = "top")
ggsave(out[["skewness"]], p, width = 22, height = 3, dpi = 150)
options(repr.plot.width = 22, repr.plot.height = 3)
p

## Summary

In [ ]:
tab <- sk[, .(pct = round(100 * mean(skew > 2), 1), med = round(median(skew), 2)), by = .(dataset, scale)]
tab <- dcast(tab, dataset ~ scale, value.var = c("pct", "med"))
setnames(tab, c("dataset", "pct_skew_gt2_linear", "pct_skew_gt2_log2", "median_skew_linear", "median_skew_log2"))
tab[, `:=`(log2_applied = log2_applied[as.character(dataset)], meanvar_slope_linear = round(slopes[as.character(dataset)], 2))]
tab <- tab[order(dataset)]
fwrite(tab, out[["summary"]])
tab